# Macclesfield vs Crystal Palace FA Cup
## 2-1

In [2]:
import json
import pandas as pd
import re

In [5]:
# Load the event types reference
event_types = pd.read_csv('opta_event_types_with_categories.csv', encoding='utf-8-sig')
print("Event types loaded:", event_types.shape)

# Load the match data (JSONP format)
with open('MacclesfieldFC_vs_CrystalPalace_event_data.json', 'r', encoding='utf-8') as f:
    jsonp_content = f.read()

# Remove the JSONP wrapper to get pure JSON
json_str = re.search(r'\({.*}\)', jsonp_content, re.DOTALL).group(0)[1:-1]
match_data = json.loads(json_str)

# Extract events into a DataFrame
events = match_data['liveData']['event']
events_df = pd.DataFrame(events)

print(f"\n=== Events DataFrame ===")
print(f"Total events: {len(events_df)}")
print(f"Columns: {events_df.columns.tolist()}")

# Join events with event types on typeId
# event_types has 'eventTypeId' and events_df has 'typeId'
df_events = events_df.merge(
    event_types,
    left_on='typeId',
    right_on='eventTypeId',
    how='left'
)

Event types loaded: (77, 6)

=== Events DataFrame ===
Total events: 1685
Columns: ['id', 'eventId', 'typeId', 'periodId', 'timeMin', 'timeSec', 'contestantId', 'outcome', 'x', 'y', 'timeStamp', 'lastModified', 'qualifier', 'playerId', 'playerName', 'keyPass', 'assist']


In [8]:
print("Columns: ",df_events.columns)
df_events.head()

Columns:  Index(['id', 'eventId', 'typeId', 'periodId', 'timeMin', 'timeSec',
       'contestantId', 'outcome', 'x', 'y', 'timeStamp', 'lastModified',
       'qualifier', 'playerId', 'playerName', 'keyPass', 'assist', 'No.',
       'eventTypeName', 'eventTypeId', 'Description', 'macro_category',
       'categorias'],
      dtype='object')


,id,eventId,typeId,periodId,timeMin,timeSec,contestantId,outcome,x,y,...,playerId,playerName,keyPass,assist,No.,eventTypeName,eventTypeId,Description,macro_category,categorias
0,2888349617,1,34,16,0,0,7s3vvybkimvx704zatm7jovtg,1,0.0,0.0,...,NaN,NaN,NaN,NaN,34.0,Team setp up,34.0,"Team line up; qualifiers 30, 44, 59, 130, 131 ...",match_admin,Gestión de partido / cambios / organización
1,2888351015,1,34,16,0,0,1c8m2ko0wxq1asfkuykurdr0y,1,0.0,0.0,...,NaN,NaN,NaN,NaN,34.0,Team setp up,34.0,"Team line up; qualifiers 30, 44, 59, 130, 131 ...",match_admin,Gestión de partido / cambios / organización
2,2888364801,2,32,1,0,0,7s3vvybkimvx704zatm7jovtg,1,0.0,0.0,...,NaN,NaN,NaN,NaN,32.0,Start,32.0,Start of a match period,match_admin,Gestión de partido / cambios / organización
3,2888364803,2,32,1,0,0,1c8m2ko0wxq1asfkuykurdr0y,1,0.0,0.0,...,NaN,NaN,NaN,NaN,32.0,Start,32.0,Start of a match period,match_admin,Gestión de partido / cambios / organización
4,2888364825,3,1,1,0,0,7s3vvybkimvx704zatm7jovtg,1,49.8,50.0,...,d62wx9jdqi6p0ygm1qcn2ddzo,J. Edmondson,NaN,NaN,1.0,Pass,1.0,Any pass attempted from one player to another ...,possession,Posesión / circulación


In [10]:
# Extract team information
teams = {team['position']: team for team in match_data['matchInfo']['contestant']}
home_team_id = teams['home']['id']
away_team_id = teams['away']['id']
home_team_name = teams['home']['name']
away_team_name = teams['away']['name']

print(f"Home: {home_team_name} (ID: {home_team_id})")
print(f"Away: {away_team_name} (ID: {away_team_id})")

# Add team names to events
df_events['team_name'] = df_events['contestantId'].map({
    home_team_id: home_team_name,
    away_team_id: away_team_name
})

# Add team position (home/away)
df_events['team_position'] = df_events['contestantId'].map({
    home_team_id: 'home',
    away_team_id: 'away'
})

Home: Macclesfield (ID: 7s3vvybkimvx704zatm7jovtg)
Away: Crystal Palace (ID: 1c8m2ko0wxq1asfkuykurdr0y)


In [11]:
print("\n=== Basic Event Statistics ===")
print(df_events.groupby('team_name')['eventId'].count().sort_values(ascending=False))


=== Basic Event Statistics ===
team_name
Crystal Palace    1010
Macclesfield       675
Name: eventId, dtype: int64


In [12]:
print("\n=== Events by Team and Category ===")
event_summary = df_events.groupby(['team_name', 'macro_category'])['eventId'].count().unstack(fill_value=0)
print(event_summary)


=== Events by Team and Category ===
macro_category  defending  dribble_duel  feed_meta  foul_card  goalkeeper  \
team_name                                                                   
Crystal Palace        121            78         25         24          12   
Macclesfield          120            85         16         24          17   

macro_category  match_admin  offside  possession  shot  stoppage_restart  
team_name                                                                 
Crystal Palace           15        7         631    13                69  
Macclesfield             18        7         287    13                69  


In [13]:
print("\n=== Key Event Types by Team ===")
key_events = df_events[df_events['eventTypeName'].isin([
    'Pass', 'Shot', 'Goal', 'Save', 'Tackle', 'Interception', 
    'Clearance', 'Aerial', 'Foul', 'Miss', 'Saved Shot'
])]
key_summary = key_events.groupby(['team_name', 'eventTypeName'])['eventId'].count().unstack(fill_value=0)
print(key_summary)

print("\n=== Final Score ===")
scores = match_data['liveData']['matchDetails']['scores']
print(f"{home_team_name}: {scores['total']['home']}")
print(f"{away_team_name}: {scores['total']['away']}")


=== Key Event Types by Team ===
eventTypeName   Aerial  Clearance  Foul  Goal  Interception  Miss  Pass  Save  \
team_name                                                                       
Crystal Palace      48         35    21     1             6     6   582     6   
Macclesfield        48         52    21     2             7     5   250     5   

eventTypeName   Saved Shot  Tackle  
team_name                           
Crystal Palace           5      16  
Macclesfield             6      20  

=== Final Score ===
Macclesfield: 2
Crystal Palace: 1
